## Importaciones y constantes

In [33]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

load_dotenv()

# Interfaz de LangChain para usar la función de embedding de Gemini
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Ruta en la que se guardarán las colecciones de Chroma
PERSIST_DIRECTORY="./chroma_db"

# Clientes de ChromaDB usando la interfaz de LangChain
# uno por cada colección (movies, people y reviews)

# vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

## Consultas a la API de TMDB

In [16]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Consultas a Wikipedia

In [66]:
import wikipedia

def search_in_wikipedia(query: str, isMovie: bool = False):
    if isMovie:
        query += " (film)"

    try:
        summary = wikipedia.summary(query)
        url = wikipedia.page(query).url
        content = wikipedia.page(query).content

        return {"page_content": summary + content, "url": url}
    except wikipedia.exceptions.DisambiguationError as e:
        # Si hay ambigüedad, elegir la opción que contenga "film" o "película"
        for option in e.options:
            if "film" in option.lower() or "película" in option.lower():
                summary = wikipedia.summary(option)
                url = wikipedia.page(option).url
                content = wikipedia.page(query).content

                return {"page_content": summary + content, "url": url}
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}
    except Exception:
        return { "page_content": None, "url": f"No se encontró información en Wikipedia para '{query}'."}

{'page_content': None,
 'url': "No se encontró información en Wikipedia para 'marilyn monroe'."}

## Añadir películas a la colección "movies" de ChromaDB

In [29]:
def add_movies_to_collection(title: str):
  """
  Busca infomación de una película a partir de su título.

  Args:
    title (str): El título de la película sobre la que buscamos información.
  """
  movies_info = get_movies_info(title)

  # Vector Store de la base de datos de ChromaDB
  vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  print("Added these movies to collection ('movies'):")

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = ""
    cast = ""

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast
  
  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }
  
  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = vector_store_movies.get(
      ids=[str(movie['id'])],
    )
    print(f"For movie {movie['title']} result: {result}")
    print(movie)

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:

      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      # Añadimos el documento con la información de la peli
      docsToAdd.append(Document(
        page_content=movie['overview'],
        metadata=get_metadata(movie, director, cast),
        id=str(movie['id'])
      ))

      print(docsToAdd)

      print(f"Added {movie['title']}.")

  if len(docsToAdd) > 0:
    vector_store_movies.add_documents(docsToAdd)

  print(f"Added {len(docsToAdd)} movies.")


## Añadir reviews a la colección "reviews" de ChromaDB

In [30]:
def add_reviews_to_collection(title: str):
  """
  Busca reviews y opiniones sobre una película en concreto.

  Args:
    title (str): El título de la película sobre la que necesitamos opiniones.
  """

  print(f"Added these reviews for the movie '{title}' to collection ('reviews'):")

  movies = get_movies_info(title)
  for movie in movies['results']:
    add_reviews_from_id(movie['id'], movie['title'])
    

def add_reviews_from_id(movie_id, movie_title):  
  vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  # Consultamos con nuestra colección a través
  # del id de la película
  result = vector_store_reviews.get(
    where={"movie_id": movie_id},
  )

  # Si NO hay reviews para esa película,
  # se intentan añadir
  if (len(result['ids']) == 0):
    reviews = get_movie_reviews(movie_id)

    def get_metadata(review):
      if review["author_details"]["rating"]:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
            "rating"  : review["author_details"]["rating"]
        }
      else:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
        }

    for review in reviews['results']:

      if review['content']:
        docsToAdd.append(Document(
          page_content=review['content'],
          metadata=get_metadata(review),
          id=str(review['id'])
        ))

    if len(docsToAdd) > 0:
      vector_store_reviews.add_documents(docsToAdd)

    print(f"Reviews for '{movie_title}': Added {len(docsToAdd)} reviews.")
  else:
    print(f"Reviews for '{movie_title}' already in database.")


## Añadir actores a la colección "people" de ChromaDB

In [ ]:
import json
import requests

def add_person_to_collection(name: str):
  """
  Busca información de una persona del cast de una película.

  Args:
    name (str): El nombre de la persona que el usuario está buscando.
  """
  # Buscamos los el nombre para encontrar ilos d
  url = f"https://api.themoviedb.org/3/search/person?query={name}&include_adult=false&language=en-US&page=1"
  r = requests.get(url, headers=TMDB_HEADERS)
  response = json.loads(r.text)

  print("Added these people to collection ('people'): ")

  if len(response['results']) == 0:
    print("quien es ese")

  for person in response['results']:
    add_person_from_id(person['id'])

def add_person_from_id(person_id):
  vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  # por si ya hemos guardado a esa persona
  result = vector_store_people.get(
    ids=[str(details['id'])]
  )

  if not result['ids']:
    # Como TMDB no tiene mucha información sobre actores,
    # usamos wikipedia para obtener el contenido.
    wikiRes=search_in_wikipedia(details['name'])
    content = wikiRes['page_content']

    if not content:
      if details['biography']:
        content = details['biography']
      else:
        content = details['name']

    # Añadir género de la persona
    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    # add_documents pide una lista, así que creamos una
    # con la persona que vamos a añadir
    docsToAdd.append(Document(
      page_content=content,
      metadata={'name': details['name'], 'department': details['known_for_department'], 'gender': gender},
      id=str(details['id'])
    ))
    vector_store_people.add_documents(docsToAdd)

    print(f"Added {details['name']}")
  else:
    print(f"{details['name']} already exists in collection")



Added these people to collection ('people'): 
Fred Astaire already exists in collection
[Document(id='3227051', metadata={'name': 'Fred Astaire Jr.', 'department': 'Acting', 'gender': 'Not set'}, page_content='Fred Astaire (born Frederick Austerlitz, May 10, 1899 – June 22, 1987) was an American dancer, actor, singer, musician, choreographer, and presenter, whose career in stage, film, and television spanned 76 years. He is widely regarded as the "greatest popular-music dancer of all time". He received an Honorary Academy Award, a BAFTA Award, three Emmy Awards, two Golden Globe Awards, and a Grammy Award.\nAs a dancer, he was known for his uncanny sense of rhythm, creativity, effortless presentation, and tireless perfectionism, which was sometimes a burden to co-workers. His dancing showed elegance, grace, originality, and precision. He drew influences from many sources, including tap, classical dance, and the elevated style of Vernon and Irene Castle. His trademark style greatly infl

## Preguntar a la base de datos vectorial

In [70]:
def query_col(query: str, collection: str):
    """
    Consulta a una colección de las disponibles (movies, reviews y people)
    con la pregunta que ha hecho el usuario, para responder con información veraz.

    Args:
        query (str): La pregunta del usuario.
        collection (str): La colección a consultar: 'movies', 'reviews' o 'people'.
    """

    # Cogemos la colección dependiendo de lo que necesitemos
    vector_store = Chroma(collection_name=collection, embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

    # Similarity search hace una búsqueda utilizando la función de embeddings del vector_store
    results = vector_store.similarity_search(
        query=query,
        k=10,
    )

    # Lo que se devolverá, un array de diccionarios que tendrán dos propiedades:
    # - page_content (str): Los documentos en Chroma, el texto.
    # - metadata (dict): Metadata de los resultados (título, votos...).
    formatted_results: list[dict] = []

    for doc in results:
        formatted_results.append({"page_content": doc.page_content, "metadata": doc.metadata})

    return formatted_results

query_col('el mago de oz','movies')


[{'page_content': 'Ellie lives in a distant city. One day, the evil witch Gingema conjured a hurricane that took Ellie and her dog Totoshka to the country of the Munchkins. To return home, Ellie and her friends the Scarecrow, the Tin Woodman and the Cowardly Lion will set off along the yellow brick road to the Emerald City in search of the Wizard who will grant their cherished wishes.',
  'metadata': {'movie_title': 'The Wizard of the Emerald City, Part I',
   'vote_average': 7.4,
   'release_date': '2025-01-01',
   'cast': 'Ekaterina Chervova, Yuri Kolokolnikov, Artur Vakha, Svetlana Khodchenkova, Vasilina Makovtseva, Dmitry Chebotarev, Sergey Epishev, Denis Vlasenko, Egor Koreshkov, Sofia Lebedeva, Yana Sekste, Aleksandra Bogdanova, Yevgeni Chumak, Aleksandr Kovrizhnykh, Zakhar Ronzhin, Timur Bokancha, Mariya Dudnik, ',
   'vote_count': 70,
   'director': '',
   'popularity': 4.4137}},
 {'page_content': 'An animated version of the classic story of a young farm-girl who is transported